# 02 — Analyse exploratoire du jeu préparé

Cette étape décrit les prix, les surfaces, les pièces et les répétitions de lignes dans le jeu préparé. Elle aide à repérer les points à traiter avant le feature engineering. Elle ne supprime aucune observation et n’entraîne aucun modèle.

L’unité reste une ligne DVF filtrée. La valeur foncière peut correspondre à une disposition couvrant plusieurs lignes ; les statistiques ci-dessous décrivent donc les lignes, pas nécessairement des prix individuels de logement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("data/processed/dvf_eligible_rows_2025.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Fichier absent. Lance d’abord python ml/preprocessing/prepare_dvf.py"
    )

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["sale_date"],
    dtype={
        "postal_code": "string",
        "department_code": "string",
        "commune_code": "string",
    },
)
print(f"Lignes : {len(df):,} — colonnes : {len(df.columns)}")
display(df.head())
display(df.dtypes.to_frame("Type de données"))

## 1. Vérifier la cohorte

On contrôle les champs requis et les catégories. Le code postal est une variable géographique facultative : quelques lignes n’en ont pas, mais cela ne remet pas en cause les critères de sélection du projet.

In [ ]:
required = ["sale_price_eur", "built_area_m2", "rooms", "property_type"]
display(df[required].isna().sum().to_frame("Valeurs manquantes"))
print("Types de logement :", df["property_type"].value_counts().to_dict())
print("Pièces égales à zéro :", int(df["rooms"].eq(0).sum()))
print("Dates distinctes :", df["sale_date"].nunique())
print("Départements distincts :", df["department_code"].nunique())
commune_count = df[["department_code", "commune_code"]].drop_duplicates().shape[0]
print("Communes distinctes (département + code commune) :", commune_count)
print("Codes postaux manquants :", int(df["postal_code"].isna().sum()))

## 2. Comparer les distributions par type

Les médianes et quantiles résistent mieux aux valeurs extrêmes que la moyenne. Les ordres de grandeur restent descriptifs tant qu’on n’a pas isolé les dispositions portant sur un seul logement.

In [ ]:
quantile_levels = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
price_quantiles = (
    df.groupby("property_type")["sale_price_eur"]
    .quantile(quantile_levels)
    .unstack()
)
area_quantiles = (
    df.groupby("property_type")["built_area_m2"]
    .quantile(quantile_levels)
    .unstack()
)
room_quantiles = (
    df.groupby("property_type")["rooms"]
    .quantile(quantile_levels)
    .unstack()
)
price_quantiles.columns = [f"p{int(q * 100)}" for q in price_quantiles.columns]
area_quantiles.columns = [f"p{int(q * 100)}" for q in area_quantiles.columns]
room_quantiles.columns = [f"p{int(q * 100)}" for q in room_quantiles.columns]
display(price_quantiles.round(0).astype("int64").rename_axis(columns="Prix en euros"))
display(area_quantiles.round(1).rename_axis(columns="Surface bâtie en m²"))
display(room_quantiles.rename_axis(columns="Nombre de pièces"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for property_type, group in df.groupby("property_type"):
    axes[0].hist(
        group["sale_price_eur"],
        bins=80,
        alpha=0.55,
        label=property_type,
    )
    axes[1].hist(
        group["built_area_m2"],
        bins=60,
        alpha=0.55,
        label=property_type,
    )
axes[0].set_xscale("log")
axes[0].set_title("Prix foncier — axe logarithmique")
axes[0].set_xlabel("Prix (€)")
axes[1].set_title("Surface bâtie")
axes[1].set_xlabel("Surface (m²)")
for axis in axes:
    axis.set_ylabel("Nombre de lignes")
    axis.legend()
fig.tight_layout()
plt.show()

## 3. Repérer les valeurs extrêmes sans les supprimer

Une valeur éloignée du reste peut être réelle, correspondre à une mutation de plusieurs biens ou signaler un problème de données. Le seuil IQR sert ici à compter et localiser le phénomène, pas à décider automatiquement qu’une ligne est fausse.

In [ ]:
outlier_rows = []
for property_type, group in df.groupby("property_type"):
    q1 = group["sale_price_eur"].quantile(0.25)
    q3 = group["sale_price_eur"].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outside = group["sale_price_eur"].lt(lower) | group["sale_price_eur"].gt(upper)
    outlier_rows.append(
        {
            "property_type": property_type,
            "iqr_lower_eur": lower,
            "iqr_upper_eur": upper,
            "rows_outside_iqr": int(outside.sum()),
            "share_outside_iqr_pct": 100 * outside.mean(),
            "max_price_eur": group["sale_price_eur"].max(),
        }
    )
display(pd.DataFrame(outlier_rows).set_index("property_type").round(1))

per_m2 = df["sale_price_eur"] / df["built_area_m2"]
print("Prix/surface — quantiles descriptifs :")
display(
    
        per_m2.groupby(df["property_type"])
        .quantile([0.01, 0.5, 0.95, 0.99])
        .unstack()
        .round(1)
    
)

## 4. Répétitions dans les variables du modèle

Les doublons stricts sur les 43 colonnes sources ont déjà été retirés. Certains enregistrements restent identiques dans les huit colonnes conservées, car leurs différences se trouvaient dans des champs source que nous avons retirés. Ces lignes sont conservées selon la règle convenue.

Pour éviter qu’un profil strictement identique soit présent à la fois dans l’entraînement et l’évaluation, on comptera les groupes d’entrées identiques lors de la séparation des données. Les codes géographiques sont utilisés ici comme variables, pas comme identifiants personnels.

In [ ]:
input_columns = [
    "sale_date",
    "property_type",
    "built_area_m2",
    "rooms",
    "postal_code",
    "department_code",
    "commune_code",
]
same_inputs = df.duplicated(subset=input_columns)
price_counts_by_input = (
    df.groupby(input_columns, dropna=False)["sale_price_eur"].nunique()
)
print(f"Lignes ayant un profil d’entrée déjà présent : {int(same_inputs.sum()):,}")
print(
    "Profils d’entrée associés à plusieurs prix : "
    f"{int(price_counts_by_input.gt(1).sum()):,}"
)
strict_duplicate_count = int(df.duplicated().sum())
print(
    "Doublons stricts dans les 8 colonnes de sortie : "
    f"{strict_duplicate_count:,}"
)

## Conclusion et suite

- Les prix sont très asymétriques : une échelle logarithmique aide à visualiser leur distribution.
- Des prix/surfaces extrêmes et des logements avec zéro pièce existent ; on les garde en l’état pendant cette analyse.
- Des profils d’entrée identiques peuvent porter des cibles différentes, et les colonnes de sortie contiennent des répétitions conservées.
- La valeur foncière reste une cible de disposition, pas nécessairement le prix individuel de chaque logement de la ligne.

Avant d’entraîner, il faudra auditer les lignes extrêmes et les dispositions à plusieurs logements, puis choisir une séparation train/validation qui garde les profils identiques dans le même groupe. Aucun filtre d’outliers n’est appliqué dans ce notebook.

## 5. Examiner la date de mutation

On compare le nombre de lignes et le prix médian par mois et par type. La médiane limite l’influence des valeurs très élevées, mais ces chiffres restent pondérés par les lignes DVF : ils ne mesurent pas nécessairement l’évolution du prix d’un logement comparable. Le fichier ne couvre ici que 2025.

In [ ]:
monthly = (
    df.assign(month=df["sale_date"].dt.to_period("M"))
    .groupby(["month", "property_type"])
    .agg(
        rows=("sale_price_eur", "size"),
        median_price_eur=("sale_price_eur", "median"),
    )
    .reset_index()
)
display(monthly.pivot(index="month", columns="property_type", values="rows"))
display(
    monthly.pivot(
        index="month", columns="property_type", values="median_price_eur"
    ).round(0)
)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for property_type, group in monthly.groupby("property_type"):
    axes[0].plot(
        group["month"].astype(str), group["median_price_eur"],
        marker="o", label=property_type,
    )
    axes[1].plot(
        group["month"].astype(str), group["rows"],
        marker="o", label=property_type,
    )
axes[0].set_ylabel("Prix médian enregistré (€)")
axes[1].set_ylabel("Nombre de lignes")
axes[1].set_xlabel("Mois de mutation en 2025")
for axis in axes:
    axis.tick_params(axis="x", rotation=45)
    axis.legend()
fig.tight_layout()
plt.show()

## 6. Bilan et décision pour la suite

- Le jeu contient 1 061 104 lignes : 605 377 maisons et 455 727 appartements. Les champs de sélection obligatoires sont complets ; 39 codes postaux manquent.
- Les distributions de prix sont très asymétriques. Le prix médian est de 210 500 € pour les maisons et 180 000 € pour les appartements. Les quantiles élevés et les maxima signalent des observations à examiner, pas à supprimer automatiquement.
- 2 012 lignes ont zéro pièce et restent incluses, conformément à la règle définie.
- 109 954 lignes sont identiques dans les huit colonnes exportées, et 8 741 profils d’entrée observés correspondent à plusieurs prix. Nous les gardons ; une future séparation des données devra éviter de répartir un même profil entre entraînement et validation.
- La date est utile pour une analyse temporelle et reste dans le jeu. Cependant, une seule année ne suffit pas à établir une tendance de long terme.
- Dans les lignes retenues, les deux identifiants documentaires sont vides. Le regroupement date + département + commune + numéro de disposition produit 124 872 groupes associés à plusieurs prix : cette combinaison ne permet donc pas d’identifier sûrement une mutation.

**Décision :** l’objectif produit est maintenant le prix d’un logement vendu seul. La cohorte candidate correspondante est construite séparément avec une règle de regroupement date + prix documentée comme approximation dans `data/README.md` et étudiée dans `notebooks/03_single_home_cohort.ipynb`. La cohorte initiale reste intacte.